<a href="https://colab.research.google.com/github/Diego-1099/Colabfiles/blob/main/Tarea_5_Representaci%C3%B3n_con_Frames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---



# **Tarea 5. Representación con Frames**

**1. Diseño de la estructura de Frames**

Para esta implementación, modelamos el ecosistema del Agente Virtual y los clientes mediante Frames.

**A) Jerarquía de Herencia (IS-A) - Mínimo 6 tipos, 2 niveles**

1. Entidad_Comercial
    - Cliente_B2B
      - Cliente_Mayorista

2. Componente_Software
    - Modulo_Conversacional
      - Base_De_Datos

**B) Jerarquía de Agregación (Parte-Todo) - 2 niveles**

    - Ecosistema_Zaro (Sistema principal) tiene_parte Agente_WhatsApp (Subsistema)
    - Agente_WhatsApp (Subsistema) tiene_parte Filtro_Belleza (Componente).

**C) Definición de Slots (Atributos) - Mínimo 8 slots**

1. estado_cuenta: [Allowed: 'al_corriente', 'moroso']
2. descuento_base: [Default: 0.0]
3. limite_credito: [Default: 5000.0]
4. saldo_pendiente: Numérico
5. riesgo_financiero: Slot calculado por un demonio _if-needed_
6. estado_sistema: [Allowed: 'activo', 'inactivo', 'mantenimiento'][Default: 'activo']
7. consultas_procesadas: Numérico (Dispara demonio _on-set_ al cambiar)
8. accion_recomendada: Llenado por reglas de inferencia.

**D) Instancias (Frames) - Mínimo 6**

1. grupo_mexicano (Instancia de Cliente_Mayorista)
2. tienda_abarrotes (Instancia de Cliente_B2B)
3. cliente_nuevo (Instancia de Cliente_B2B)
4. agente_whatsapp (Instancia de Modulo_Conversacional)
5. filtro_belleza (Instancia de Modulo_Conversacional)
6. memoria_elefante (Instancia de Base_De_Datos)



---



# **Paso 2. Implementación en Python (Motor de Frames)**

In [ ]:
# ==========================================
# 1. DEFINICIÓN DEL MOTOR DE FRAMES
# ==========================================

class Slot:
  def __init__(self, name, default = None, allowed = None, if_needed = None, on_set = None):
    self.name = name
    self.value = None
    self.default = default
    self.allowed = allowed
    self.if_needed = if_needed
    self.on_set = on_set

class Frame:
  def __init__(self, name, is_a = None):
    self.name = name
    self.is_a = is_a
    self.slots = {}
    self.partes = []
    self.parte_de = None

  def add_slot (self, slot):
    self.slots[slot.name] = slot

  def set_value (self, slot_name, val):
    slot = self._get_slot_obj(slot_name)
    if not slot:
      print(f"Error: Slot '{slot_name}' no existe en {self.name}.")
      return

    if slot.allowed and val not in slot.allowed:
      print(f"Error: Valor '{val}' no permitido para '{slot_name}'. Permitidos: {slot.allowed}")
      return

    slot.value = val

    if slot.on_set:
      slot.on_set(self, val)

  def get_value (self, slot_name):
    slot = self._get_slot_obj(slot_name)
    if not slot:
      return None

    if slot.value is not None:
      return slot.value

    if slot.if_needed:
      return slot.if_needed(self)

    return slot.default

  def _get_slot_obj (self, slot_name):
    if slot_name in self.slots:
      return self.slots[slot_name]
    if self.is_a:
      return self.is_a._get_slot_obj(slot_name)
    return None

  def add_parte (self, componente):
    self.partes.append(componente)
    componente.parte_de = self

def is_a_check (frame, super_frame_name):
  actual = frame
  while actual:
    if actual.name == super_frame_name:
      return True
    actual = actual.is_a
  return False

# ==========================================
# 2. DEFINICIÓN DE DEMONIOS Y REGLAS
# ==========================================

def calc_riesgo_financiero(frame):
    saldo = frame.get_value("saldo_pendiente") or 0
    limite = frame.get_value("limite_credito") or 1
    print(f"[Demonio IF-NEEDED ejecutado] Calculando riesgo para {frame.name}...")
    if saldo > limite:
        return "Critico"
    elif saldo > 0:
        return "Medio"
    return "Bajo"

def log_consultas(frame, nuevo_valor):
    print(f"[Demonio ON-SET ejecutado] {frame.name} ha procesado {nuevo_valor} consultas. Actualizando métricas de rendimiento...")

def infer(frame):
    cambios = 0
    print(f"\n--- Iniciando inferencia para: {frame.name} ---")

    # Regla 1: Infiere etiqueta de estado
    riesgo = frame.get_value("riesgo_financiero")
    estado_actual = frame.get_value("estado_cuenta")
    if riesgo == "Critico" and estado_actual != "moroso":
        print(f"-> Regla 1 disparada: Riesgo Crítico. Cambiando estado_cuenta a 'moroso'.")
        frame.set_value("estado_cuenta", "moroso")
        cambios += 1

    # Regla 2: Infiere acción/recomendación
    if frame.get_value("estado_cuenta") == "moroso":
        print(f"-> Regla 2 disparada: Cliente moroso. Acción: 'bloquear_ventas_y_notificar'.")
        frame.set_value("accion_recomendada", "bloquear_ventas_y_notificar")
        cambios += 1
    elif riesgo == "Bajo":
        print(f"-> Regla 2 disparada: Cliente sano. Acción: 'ofrecer_promociones'.")
        frame.set_value("accion_recomendada", "ofrecer_promociones")
        cambios += 1

    if cambios == 0:
        print("Ninguna regla aplicó.")

# ==========================================
# 3. CREACIÓN DE TIPOS E INSTANCIAS
# ==========================================

# Tipos (IS-A)
entidad_comercial = Frame("Entidad_Comercial")
entidad_comercial.add_slot(Slot("limite_credito", default = 5000.0))
entidad_comercial.add_slot(Slot("saldo_pendiente", default = 0.0))

cliente_b2b = Frame("Cliente_B2B", is_a = entidad_comercial)
cliente_b2b.add_slot(Slot("estado_cuenta", allowed = ["al_corriente", "moroso"]))
cliente_b2b.add_slot(Slot("riesgo_financiero", if_needed = calc_riesgo_financiero))
cliente_b2b.add_slot(Slot("accion_recomendada"))

cliente_mayorista = Frame("Cliente_Mayorista", is_a = cliente_b2b)
cliente_mayorista.add_slot(Slot("descuento_base", default = 15.0))

componente_software = Frame("Componente_Software")
componente_software.add_slot(Slot("estado_sistema", default = "activo", allowed = ["activo", "inactivo", "mantenimiento"]))

modulo_conversacional = Frame("Modulo_Conversacional", is_a = componente_software)
modulo_conversacional.add_slot(Slot("consultas_procesadas", default = 0, on_set = log_consultas))

# Instancias
grupo_mexicano = Frame("grupo_mexicano", is_a = cliente_mayorista)
tienda_abarrotes = Frame("tienda_abarrotes", is_a = cliente_b2b)
agente_whatsapp = Frame("agente_whatsapp", is_a = modulo_conversacional)
filtro_belleza = Frame("filtro_belleza", is_a = modulo_conversacional)
ecosistema_zaro = Frame("ecosistema_zaro", is_a = componente_software)

# Agregación (Parte-Todo)
ecosistema_zaro.add_parte(agente_whatsapp)
agente_whatsapp.add_parte(filtro_belleza)


# ==========================================
# 4. PRUEBAS MÍNIMAS (CHECKLIST DE LA TAREA)
# ==========================================

print("--- CHECKLIST DE PRUEBAS ---")

print("\n1. Herencia y jerarquía IS-A (is_a_check):")
print(f"¿grupo_mexicano es_un Entidad_Comercial?: {is_a_check(grupo_mexicano, 'Entidad_Comercial')}")

print("\n2. get() de slot heredado con default:")
print(f"grupo_mexicano hereda descuento_base: {grupo_mexicano.get_value('descuento_base')}%")
print(f"tienda_abarrotes hereda limite_credito: ${tienda_abarrotes.get_value('limite_credito')}")

print("\n3. Agregación (tiene_parte / parte_de):")
print(f"Partes de ecosistema_zaro: {[p.name for p in ecosistema_zaro.partes]}")
print(f"Partes de agente_whatsapp: {[p.name for p in agente_whatsapp.partes]}")
print(f"¿filtro_belleza es parte_de?: {filtro_belleza.parte_de.name}")

print("\n4. Demonio ON-SET:")
agente_whatsapp.set_value("consultas_procesadas", 150)

print("\n5. Demonio IF-NEEDED y validación ALLOWED:")
# Forzamos un saldo crítico para probar reglas más adelante
tienda_abarrotes.set_value("saldo_pendiente", 8000.0)
tienda_abarrotes.set_value("estado_cuenta", "al_corriente")
tienda_abarrotes.set_value("estado_cuenta", "con_deuda")
print(f"Riesgo de tienda_abarrotes: {tienda_abarrotes.get_value('riesgo_financiero')}")

print("\n6. Inferencia por reglas (infer()):")
infer(tienda_abarrotes)
print(f"Estado final tienda_abarrotes -> Cuenta: {tienda_abarrotes.get_value('estado_cuenta')}, Acción: {tienda_abarrotes.get_value('accion_recomendada')}")

infer(grupo_mexicano)
print(f"Estado final grupo_mexicano -> Acción: {grupo_mexicano.get_value('accion_recomendada')}")

--- CHECKLIST DE PRUEBAS ---

1. Herencia y jerarquía IS-A (is_a_check):
¿grupo_mexicano es_un Entidad_Comercial?: True

2. get() de slot heredado con default:
grupo_mexicano hereda descuento_base: 15.0%
tienda_abarrotes hereda limite_credito: $5000.0

3. Agregación (tiene_parte / parte_de):
Partes de ecosistema_zaro: ['agente_whatsapp']
Partes de agente_whatsapp: ['filtro_belleza']
¿filtro_belleza es parte_de?: agente_whatsapp

4. Demonio ON-SET:
[Demonio ON-SET ejecutado] agente_whatsapp ha procesado 150 consultas. Actualizando métricas de rendimiento...

5. Demonio IF-NEEDED y validación ALLOWED:
Error: Valor 'con_deuda' no permitido para 'estado_cuenta'. Permitidos: ['al_corriente', 'moroso']
[Demonio IF-NEEDED ejecutado] Calculando riesgo para tienda_abarrotes...
Riesgo de tienda_abarrotes: Critico

6. Inferencia por reglas (infer()):

--- Iniciando inferencia para: tienda_abarrotes ---
[Demonio IF-NEEDED ejecutado] Calculando riesgo para tienda_abarrotes...
-> Regla 1 disparada: 



---



# **Paso 3. Diagrama Simple de la Estructura de Frames**

In [ ]:
'''
=====================================================
                 1. JERARQUÍA IS-A
=====================================================
Entidad_Comercial
 ├── Cliente_B2B
 │    └── Cliente_Mayorista
 │
Componente_Software
 ├── Modulo_Conversacional
 └── Base_De_Datos

=====================================================
             2. JERARQUÍA PARTE-TODO
=====================================================
Ecosistema_Zaro
 └── Agente_WhatsApp
      └── Filtro_Belleza

=====================================================
           3. DEMONIOS Y REGLAS ACTIVOS
=====================================================
[Demonios]
 1. calc_riesgo_financiero (IF-NEEDED):
    Calcula dinámicamente si el saldo supera el límite.
 2. log_consultas (ON-SET):
    Registra métricas al actualizar el contador de consultas.

[Reglas de Inferencia]
 1. Regla de Morosidad:
    SI riesgo == 'Critico' ENTONCES estado_cuenta = 'moroso'.
 2. Regla de Acción Comercial:
    SI estado == 'moroso' ENTONCES accion = 'bloquear_ventas'
    SI riesgo == 'Bajo' ENTONCES accion = 'ofrecer_promociones'
'''



---



# **Paso 4. Preguntas Reflexivas**

**1. Ontología implícita del dominio (compromisos ontológicos):**

Consideré como entidades "reales" a los actores físicos / jurídicos (Clientes) y a los módulos de software (Agente, Filtro). Decidí no representar las "Ventas" como entidades (frames), sino como eventos transitorios manejados por el motor.

  - Decisión 1 (Entidad): Modelé Cliente_Mayorista como entidad para heredar propiedades complejas. Gané estructuración, perdí simplicidad.
  - Decisión 2 (Slot): Modelé el riesgo_financiero como un slot calculado por un demonio en lugar de un objeto. Gané dinamismo (se calcula en tiempo real), perdí la capacidad de darle atributos propios al riesgo.
  - Decisión 3 (Relación): Modelé la pertenencia del Filtro de Belleza al Agente como relación de agregación. Gané capacidad de hacer consultas estructurales de dependencias.

**2. IS-A vs Parte-todo:**

  - Caso 1: ¿El Agente_WhatsApp es un tipo de Ecosistema_Zaro o es parte de él? Elegí parte-todo porque el ecosistema es el todo funcional; el agente no hereda las propiedades del ecosistema, sino que es un engranaje. Esto habilita inferencias estructurales (si el ecosistema se apaga, las partes se apagan).

  - Caso 2: ¿Cliente_Mayorista es parte de un Cliente_B2B o es un tipo de? Elegí IS-A porque el mayorista comparte la misma naturaleza ontológica y debe heredar el límite de crédito y el estado de cuenta.

**3. Granularidad y fronteras del frame:**
El criterio fue la utilidad para la toma de decisiones comerciales.
  - Subdivisión (Más granular): Dividí Componente_Software en Modulo_Conversacional y Base_De_Datos. Impacto: permite explicar por qué la "Memoria de Elefante" guarda datos pero no procesa lenguaje natural.
  - Unificación (Menos granular): Dejé estado_cuenta como un simple slot de texto en lugar de crear un frame complejo de "HistorialCrediticio". Impacto: facilita la inferencia rápida de las reglas (bloquear o no bloquear ventas) sin hacer un recorrido profundo, aunque se pierde el detalle de qué factura exacta se debe.

**4. Defaults, excepciones y razonamiento no monotónico:**

  - Default 1: limite_credito = 5000.0. Representa lo "típico" para un cliente B2B estándar de primera entrada.
  - Default 2: estado_sistema = 'activo'. Representa que el software, por naturaleza, debe estar corriendo.
  - Excepción: Grupo Mexicano es un cliente corporativo masivo, 5,000 de límite es irrisorio. La representación lo maneja haciendo un override (sobreescritura) del slot limite_credito directamente en su instancia a 500,000.
  - Riesgo: El razonamiento no monotónico con defaults corre el riesgo de asumir cosas falsas temporalmente. Si un cliente nuevo es en realidad una empresa fantasma de alto riesgo, el sistema asumirá ingenuamente que tiene 5,000 de crédito limpio hasta que una validación externa actualice el slot, exponiendo a la empresa a un fraude inicial.
  




---

